# RouteRadar — Historical TfL data profiling

**Purpose:** Investigate the source data *before* deciding how to clean and model it.

This notebook explores the original TfL service-status records and records evidence for the engineering decisions in RouteRadar. It **does not modify the original CSV or the database**.

**Questions to answer**
1. What is the structure and coverage of the source?
2. Which fields are missing or inconsistent?
3. Are timestamps valid, and which intervals are suspicious?
4. How many rows are exact duplicates?
5. Which service-status values and Underground line labels exist?
6. How often do status-message intervals overlap or touch?
7. Can we identify individual trains or incidents? (Spoiler: This is not possible.)

## Setup: connect to the existing raw data
First run `python3 src/load_historical.py` from the project root. 

In [85]:
# connect to the database
from pathlib import Path
import duckdb
import pandas as pd

# project directory
ROOT = Path.cwd()

data_folder = ROOT/"data"
if not data_folder.exists():
    ROOT = ROOT.parent

DB_PATH = ROOT /"data"/"processed"/"routeradar.duckdb"

# open database connection
conn = duckdb.connect(str(DB_PATH), read_only = True)

# inspect the raw table
conn.execute("""
SELECT *
FROM raw_service_status
LIMIT 5
""").df()



,Line,Start Date/Time,End Date/Time,Service Status,Status Message Details
0,Northern,01/04/2022 04:25:01,01/04/2022 04:27:46,Part Closure,No service between Moorgate and Kennington due...
1,Northern,01/04/2022 04:27:46,01/04/2022 11:01:45,Part Closure,No service between Moorgate and Kennington due...
2,Bakerloo,01/04/2022 05:19:49,01/04/2022 05:28:20,Part Suspended,No service between Queens Park and Harrow & We...
3,Bakerloo,01/04/2022 05:28:20,01/04/2022 05:41:53,Part Suspended,No service between Queens Park and Harrow & We...
4,Bakerloo,01/04/2022 05:41:53,01/04/2022 05:42:00,Part Suspended,No service between Queens Park and Harrow & We...


## 1. Understanding the source dataset
**Question:** What columns exist, what are their types, and what does a typical row look like? I imported the CSV columns as text so they could be inspected before parsing.

In [62]:
raw_service_status_describle = conn.execute("""
DESCRIBE raw_service_status
""").df()

display(raw_service_status_describle)

,column_name,column_type,null,key,default,extra
0,Line,VARCHAR,YES,None,None,None
1,Start Date/Time,VARCHAR,YES,None,None,None
2,End Date/Time,VARCHAR,YES,None,None,None
3,Service Status,VARCHAR,YES,None,None,None
4,Status Message Details,VARCHAR,YES,None,None,None


In [61]:
raw_service_status_inspect = conn.execute("""
SELECT * FROM raw_service_status
LIMIT 5
""").df()

display(raw_service_status_inspect)

,Line,Start Date/Time,End Date/Time,Service Status,Status Message Details
0,Northern,01/04/2022 04:25:01,01/04/2022 04:27:46,Part Closure,No service between Moorgate and Kennington due...
1,Northern,01/04/2022 04:27:46,01/04/2022 11:01:45,Part Closure,No service between Moorgate and Kennington due...
2,Bakerloo,01/04/2022 05:19:49,01/04/2022 05:28:20,Part Suspended,No service between Queens Park and Harrow & We...
3,Bakerloo,01/04/2022 05:28:20,01/04/2022 05:41:53,Part Suspended,No service between Queens Park and Harrow & We...
4,Bakerloo,01/04/2022 05:41:53,01/04/2022 05:42:00,Part Suspended,No service between Queens Park and Harrow & We...


In [ ]:
# How many lines are there?
train_lines = conn.execute("""
SELECT DISTINCT(line)
FROM raw_service_status
ORDER BY line
""").df()

display(train_lines)

,Line
0,Bakerloo
1,C&H
2,Central
3,District
4,Jubilee
5,Metropolitan
6,Northern
7,Piccadilly
8,Victoria
9,Waterloo & City


In [79]:
service_statuss = conn.execute("""
SELECT DISTINCT "Service Status" AS service_status
FROM raw_service_status
ORDER BY service_status
""").df()

display(service_statuss)

,service_status
0,Good Service
1,Minor Delays
2,Part Closure
3,Part Suspended
4,Planned Closure
5,Reduced Service
6,Service Closed
7,Severe Delays
8,Special Service
9,Suspended


**Observation:** There are exactly 10 distinct lines and 10 distinct service status'. The C&H label required further investigation before mapping the source labels to individual undergraound lines.

In [76]:
# How many lines are there?
status_message_detailss = conn.execute("""
SELECT DISTINCT TRIM("Status Message Details") AS status_message_details
FROM raw_service_status
ORDER BY status_message_details
""").df()

display(status_message_detailss)

,status_message_details
0,\r\n There will be no service between Wood Gre...
1,\r\nService will resume at 06:00.
2,\r\nService will resume later this morning how...
3,\r\nService will resume later this morning.
4,\r\nSevere delays due to a shortage of trains....
...,...
46155,train service will resume at 06:00
46156,train service will resume later this morning
46157,train services will resume later this morning.
46158,while the emergency services deal with a casua...


**Observation:** There are many different status_message_details even after accounting for blank spaces at the begining and end of the text. Many of them are similar, meaning the same thing. Small differences can be things like a full stop, a capitalised letter or line breaks such as '\r\n'.

**Potential decision:** There is no train ID or disruption ID supplied in the CSV. If disruption level analysis is required, related messages may need to be grouped up using inferred relationships based on line, timestamps and status message details. These results would be estimates rather than confirmed disruptions - this should be made clear.

## 2. Record counts and historical coverage
**Question:** How much data do we have? Which dates/years appear?

In [38]:

yearly = conn.execute("""
WITH parsed AS(
SELECT TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time
FROM raw_service_status
)

SELECT YEAR(start_time) AS year,
COUNT(*) AS count
FROM parsed

WHERE start_time IS NOT NULL
GROUP BY year
""").df()

display(yearly)

,year,count
0,2023,22614
1,2025,28544
2,2026,25767
3,2022,16878
4,2024,24792


**Interpretation:** The count split by year seems to be roughly even, although 2022 count is significantly lower than the others. This could be reflect partial coverage during that year.

In [86]:
# looking at monthly counts
monthly_2022 = conn.execute("""
WITH parsed AS(
SELECT TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time
FROM raw_service_status
)

SELECT MONTH(start_time) AS month_number,
COUNT(*) AS count

FROM parsed

WHERE YEAR(start_time) = 2022

GROUP BY month_number
ORDER BY month_number ASC
""").df()

display(monthly_2022)

,month_number,count
0,4,2131
1,5,1719
2,6,2332
3,7,1671
4,8,2003
5,9,1471
6,10,1635
7,11,1611
8,12,2305


**Interpretation:** There looking to be no missing months in the 2022 dataset. The equivalent check can be done with other years - I don't show it here to avoid clutter.

## 3. Data Quality Assessment - Missing values and blank text
**Question:** Are there any absent fields? Both `NULL` and empty/whitespace strings can represent missing values.

In [20]:
quality = conn.execute("""
SELECT COUNT(*) AS total_records,

COUNT(*) FILTER(
WHERE "line" IS NULL
OR TRIM("line") = ''
) AS missing_line,

COUNT(*) FILTER(
WHERE TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') IS NULL
OR TRIM("Start Date/Time") = ''
) AS missing_start_dates,

COUNT(*) FILTER(
WHERE TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') IS NULL
OR TRIM("End Date/Time") = ''
) AS missing_end_dates,

COUNT(*) FILTER(
WHERE "Service Status" IS NULL
OR TRIM("Service Status") = '' 
) AS missing_service_status,


COUNT(*) FILTER(
WHERE "Status Message Details" IS NULL
OR TRIM("Status Message Details") = ''
) AS missing_status_message_details,

FROM raw_service_status
""").df()

display(quality)

,total_records,missing_line,missing_start_dates,missing_end_dates,missing_service_status,missing_status_message_details
0,118595,0,0,0,0,3467


**Observation:** There are zero missing lines, missing start/end dates or service status'. The only column with missing fields is status message details.

**Potential decision:** A missing 'Status Message Details' does not automatically invalidate an otherwise usable line, status or interval. 
Fields such as line, status, start, end require much stricter validation.

## 4. Invalid timestamps, zero durations and reversed intervals
**Question:** Can all timestamps be parsed? Are there intervals which finish before their start?

The dataset uses **day/month/year hour:minute:second**. `TRY_STRPTIME` returns `NULL` instead of stopping when a value cannot be parsed.

In [82]:
timestamp_inconsistancies = conn.execute("""
WITH parsed AS(
SELECT
TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time,
TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') AS end_time
FROM raw_service_status
)

SELECT
COUNT(*) FILTER (
WHERE start_time IS NULL
) AS invalid_start,

COUNT(*) FILTER (
WHERE end_time IS NULL
) AS invalid_end,

COUNT(*) FILTER (
WHERE start_time IS NOT NULL
AND end_time IS NOT NULL
AND end_time = start_time
) AS zero_time,

COUNT(*) FILTER (
WHERE start_time IS NOT NULL
AND end_time IS NOT NULL
AND end_time < start_time
) AS negative_duration,

COUNT(*) FILTER (
WHERE start_time IS NULL
OR end_time IS NULL
OR end_time <= start_time
) AS total_unusable_interval

FROM parsed
""").df()

display(timestamp_inconsistancies)

,invalid_start,invalid_end,zero_time,negative_duration,total_unusable_interval
0,0,0,43,15,58


**Potential decision:** Keep the questionable rows in the staging tables but exclude unparable and non positive intervals from the duration calculations. A zero-length status might represent an actual event so do not erase it from all analyses.

**Timezone note:** The timestamps in the dataset have no explicit timezone. I will have to verify whether TfL defines them as local London time and accounts for things such as daylight savings. I will not make any assumptions about this.

## 5. Duplicate records
**Question:** How many rows have the exact same five values? A record which is a copy of an identical row is an *exact duplicate*. two similar records with different timestamps or wording are not exact duplicates - this distinction matters.

In [42]:
duplicates = conn.execute("""
SELECT "Line",
"Start Date/Time",
"End Date/Time",
"Service Status",
"Status Message Details",
COUNT(*) AS occurrences

FROM raw_service_status

GROUP BY ALL
HAVING COUNT(*)>1

ORDER BY occurrences DESC
LIMIT 5
""").df()

display(duplicates)

,Line,Start Date/Time,End Date/Time,Service Status,Status Message Details,occurrences
0,District,23/01/2023 17:54:19,23/01/2023 17:58:42,Part Suspended,"No service between Richmond, Ealing Broadway a...",12
1,District,25/01/2026 14:49:42,25/01/2026 15:36:08,Minor Delays,No service between Edgware Road and High Stree...,9
2,District,25/01/2026 14:49:15,25/01/2026 14:49:42,Minor Delays,No service between Earl's Court and Kensington...,9
3,District,25/01/2026 14:36:29,25/01/2026 14:49:14,Minor Delays,No service between Earl's Court and Kensington...,9
4,District,23/01/2023 17:53:57,23/01/2023 17:54:19,Part Suspended,"No service between Richmond, Ealing Broadway a...",8


In [47]:
## additional insights - duplicate statistics
duplicate_stats = conn.execute("""
WITH grouped AS (
SELECT "Line",
"Start Date/Time",
"End Date/Time",
"Service Status",
"Status Message Details",
COUNT(*) AS occurrences
FROM raw_service_status
GROUP BY ALL
)
SELECT
SUM(occurrences) AS original_rows,
COUNT(*) AS unique_rows,
SUM(occurrences - 1) AS extra_duplicate_copies,
COUNT(*) FILTER (WHERE occurrences > 1) AS duplicated_groups
FROM grouped
""").df()

display(duplicate_stats)

,original_rows,unique_rows,extra_duplicate_copies,duplicated_groups
0,118595.0,104600,13995.0,10209


**Potential decision:** Use exact deduplification for calculations while keeping the original raw rows. Keep a record of excluded copies. This is completely different from merging overlapping *time duration* and different from assigning a shared *disruption ID*. 

## 6. Overlapping and touching intervals
**Question:** How often do different source messages cover the same time on the same line?

Two ranges **overlap** when the later record starts before an earlier record has finished. They **touch** when one starts exactly when an earlier one ends. Touching is not a data quality error.

To investigate, first remove exact duplicates and invalid intervals only within the query. Then compare each start time against the largest end time of all prior rows on that line

There are many occurences of exact repeating rows. The highest record has been repeated 12 times.
Here I make sure to remove exact duplicates to make sure I examine genuine interval relationships rather than comparing copies of the exact same row.

In [57]:
overlap_stats2 = conn.execute("""
WITH parsed AS(
SELECT DISTINCT
TRIM("Line") AS line,
TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time,
TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') AS end_time,
"Service Status" AS service_status,
"Status Message details" AS status_message

FROM raw_service_status
),

valid AS(
SELECT *
FROM parsed
WHERE start_time IS NOT NULL
AND line IS NOT NULL
AND end_time IS NOT NULL
AND end_time > start_time
),

ordered AS(
SELECT *,
MAX(end_time) OVER(
PARTITION BY line
ORDER BY start_time, end_time
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
) AS previous_max_end
FROM valid
)

SELECT
COUNT(*) AS valid_unique_records,

COUNT(*) FILTER(
WHERE start_time < previous_max_end
) AS overlapping_interval_records,

COUNT(*) FILTER(
WHERE start_time = previous_max_end
) AS touching_interval_records,

COUNT(*) FILTER(
WHERE previous_max_end IS NULL
) AS first_record_per_line

FROM ordered
""").df()

display(overlap_stats2)

,valid_unique_records,overlapping_interval_records,touching_interval_records,first_record_per_line
0,104544,27774,46787,10


In [52]:
overlaps = conn.execute("""
WITH parsed AS(
SELECT DISTINCT
TRIM("Line") AS line,
TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time,
TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') AS end_time,
"Service Status" AS service_status,
"Status Message details" AS status_message

FROM raw_service_status
),

valid AS(
SELECT *
FROM parsed
WHERE start_time IS NOT NULL
AND end_time > start_time
),

ordered AS(
SELECT *,
MAX(end_time) OVER(
PARTITION BY line
ORDER BY start_time, end_time
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
) AS previous_max_end
FROM valid
)

SELECT
line,
start_time,
end_time,
previous_max_end,
status_message

FROM ordered
WHERE start_time < previous_max_end

ORDER BY line, start_time
LIMIT 5
""").df()

display(overlaps)

,line,start_time,end_time,previous_max_end,status_message
0,Bakerloo,2022-04-04 10:54:08,2022-04-04 11:03:14,2022-04-04 11:03:14,No service between Queen's Park and Harrow & W...
1,Bakerloo,2022-04-04 11:03:14,2022-04-04 11:16:28,2022-04-04 11:16:28,No service between Queen's Park and Harrow & W...
2,Bakerloo,2022-04-10 16:32:54,2022-04-10 17:10:10,2022-04-10 17:10:09,No service between Queen's Park and Harrow & W...
3,Bakerloo,2022-04-23 11:13:19,2022-04-23 14:19:10,2022-04-23 14:19:08,Severe delays between Queen's Park and Harrow ...
4,Bakerloo,2022-04-23 20:04:12,2022-04-23 20:11:12,2022-04-23 20:11:12,No service between Queen's Park and Harrow & W...


The historical TfL dataset does not contain unique incident or train identifiers. This means that multiple records with similar descriptions and overlapping or consecutive timestamps may refer to the same underlying disruption. This cannot be confirmed from the data alone.

**Decision:** For RouteRadar, I will distinguish between exact duplicate records, potentially related status messages and overlapping disruption intervals.

Exact duplicate records will be removed from the analytical layer. Overlapping intervals will be merged for calculating the duration of the reported line disruption, without assuming that the associated messages represent the same disruption.

This approach avoids double counting the disruption duration while also preserving the distinction between reported status and individual disruption.

In [83]:
# close the connectection
conn.close()